<a href="https://colab.research.google.com/github/kulanigayathree/J26-IT-364/blob/model%2FFishing-Zone-Prediction/notebooks/C2_fishing_zone_prediction/C2_3_model_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# C2 – Notebook 3: Model demo page (Gradio)

A small web page to test the trained model and show it in presentations.
Pick a date and lead day → see the **probability map**, **uncertainty map**, **what actually happened**
and the **sea-surface temperature** that day.

**Needs:** Notebook 2 finished at least once (it saves `checkpoints/best.pt` and `results/normalisation.json`).

`Runtime → Change runtime type → T4 GPU` (CPU also works, just slower), then `Runtime → Run all`.
A link appears at the bottom: click it to open the page. The `gradio.live` link also works on your phone
or your supervisor's laptop for 72 hours while this notebook is running.

## Step 1 – Connect Drive and install Gradio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/C2_data'
!pip install -q gradio

Mounted at /content/drive


## Step 2 – Load the datacube, normalisation and trained model

In [ ]:
import json, numpy as np, pandas as pd, xarray as xr, torch, torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
T_IN, LEADS, DROPOUT = 7, 3, 0.2          # must match the settings used in Notebook 2

norm = json.load(open(f'{BASE}/results/normalisation.json'))
FEATURES = norm['features']
mean = np.array(norm['mean'], 'float32')[None, :, None, None]
std  = np.array(norm['std'],  'float32')[None, :, None, None]

X = xr.open_dataset(f'{BASE}/datacube_0p25.nc').load()
days = pd.DatetimeIndex(X.time.values)
ocean = X['depth'].isel(time=0).notnull().values
feat = (np.stack([X[v].values for v in FEATURES], axis=1).astype('float32') - mean) / std
feat = np.nan_to_num(feat, nan=0.0)
label = np.nan_to_num(X['fishing'].values).astype('float32')
lat, lon = X.latitude.values, X.longitude.values
C = feat.shape[1]
print('Data loaded:', days[0].date(), 'to', days[-1].date())

Data loaded: 2017-01-01 to 2024-12-31


In [ ]:
# Same model definition as Notebook 2
class ConvLSTMCell(nn.Module):
    def __init__(self, cin, ch, k=3):
        super().__init__(); self.ch = ch
        self.conv = nn.Conv2d(cin + ch, 4 * ch, k, padding=k // 2)
    def forward(self, x, h, c):
        i, f, o, g = torch.chunk(self.conv(torch.cat([x, h], 1)), 4, 1)
        c = torch.sigmoid(f) * c + torch.sigmoid(i) * torch.tanh(g)
        h = torch.sigmoid(o) * torch.tanh(c)
        return h, c

class FishZoneNet(nn.Module):
    def __init__(self, cin, hidden=(32, 32), p=DROPOUT, out=LEADS):
        super().__init__()
        self.cells = nn.ModuleList(); prev = cin
        for ch in hidden:
            self.cells.append(ConvLSTMCell(prev, ch)); prev = ch
        self.drop = nn.Dropout2d(p)
        self.head = nn.Sequential(nn.Conv2d(prev, 32, 3, padding=1), nn.ReLU(),
                                  nn.Dropout2d(p), nn.Conv2d(32, out, 1))
    def forward(self, x):
        B, T, _, H, W = x.shape
        state = [(x.new_zeros(B, c.ch, H, W), x.new_zeros(B, c.ch, H, W)) for c in self.cells]
        for t in range(T):
            inp = x[:, t]
            for j, cell in enumerate(self.cells):
                h, c = cell(inp, *state[j]); state[j] = (h, c)
                inp = self.drop(h) if j < len(self.cells) - 1 else h
        return self.head(self.drop(inp))

model = FishZoneNet(C).to(DEVICE)
model.load_state_dict(torch.load(f'{BASE}/checkpoints/best.pt', map_location=DEVICE))

# Same probability correction as Notebook 2 (undo the class weighting used in training)
tr = (days >= '2017-01-01') & (days <= '2021-12-31')
pos = label[tr][:, ocean].mean()
LOG_W = float(np.log(min((1 - pos) / max(pos, 1e-4), 50)))
print('Model loaded on', DEVICE)

Model loaded on cuda


## Step 3 – Prediction function

In [ ]:
@torch.no_grad()
def forecast(target_date, lead_day, mc_runs):
    # Predict fishing probability for target_date, made lead_day days in advance
    try:
        target = pd.Timestamp(str(target_date)[:10])
    except Exception:
        raise ValueError('Enter a date like 2023-06-15')
    lead = int(str(lead_day).split()[-1]) if not isinstance(lead_day, int) else lead_day
    if target not in days:
        raise ValueError(f'Date must be between {days[0].date()} and {days[-1].date()}')
    t = days.get_loc(target) - lead            # last input day
    if t < T_IN - 1:
        raise ValueError('Date too early: the model needs 7 earlier days of data')
    x = torch.from_numpy(feat[t - T_IN + 1: t + 1]).unsqueeze(0).to(DEVICE)
    model.train()                              # keep dropout ON for MC dropout
    ps = torch.stack([torch.sigmoid(model(x) - LOG_W)[0, lead - 1] for _ in range(int(mc_runs))])
    return ps.mean(0).cpu().numpy(), ps.std(0).cpu().numpy(), label[t + lead], t

def sea(a): return np.where(ocean, a, np.nan)

def run_demo(target_date, lead_day, mc_runs):
    p, u, y, t = forecast(target_date, lead_day, mc_runs)
    fig, ax = plt.subplots(1, 4, figsize=(20, 4.6))
    panels = [(sea(p), 'Fishing probability', 'viridis', (0, max(0.05, np.nanmax(sea(p))))),
              (sea(u), 'Uncertainty (MC std)', 'magma', None),
              (sea(y), 'Actual fishing (GFW)', 'Greys', (0, 1)),
              (sea(X['sst'].values[t]), f'SST on input day {days[t].date()} (°C)', 'coolwarm', None)]
    for a, (arr, title, cmap, lim) in zip(ax, panels):
        kw = {} if lim is None else {'vmin': lim[0], 'vmax': lim[1]}
        im = a.pcolormesh(lon, lat, arr, cmap=cmap, shading='auto', **kw)
        a.set_title(title, fontsize=11); a.set_aspect('equal'); plt.colorbar(im, ax=a, shrink=0.75)
    fig.suptitle(f'Forecast for {pd.Timestamp(str(target_date)[:10]).date()}  (made {lead_day} ahead)', fontsize=13)
    plt.tight_layout()

    yy, pp = y[ocean], p[ocean]
    top = pp >= np.quantile(pp, 0.9)                    # the 10% of cells the model likes most
    stats = {'Sea cells with real fishing': int(yy.sum()),
             'Average probability': round(float(pp.mean()), 4),
             'Average uncertainty': round(float(u[ocean].mean()), 4),
             'Fishing rate in top-10% cells': round(float(yy[top].mean()), 4),
             'Fishing rate in all cells': round(float(yy.mean()), 4),
             'ROC-AUC for this day': round(float(roc_auc_score(yy, pp)), 4) if 0 < yy.sum() < len(yy) else 'n/a'}
    return fig, pd.DataFrame(stats.items(), columns=['Measure', 'Value'])

## Step 4 – Launch the demo page
If *Fishing rate in top-10% cells* is clearly higher than *Fishing rate in all cells*, the model is pointing
to the right places. Stop the page with the ■ stop button on this cell.

In [ ]:
import gradio as gr
with gr.Blocks(title='C2 Fishing-zone forecast demo') as demo:
    gr.Markdown('## C2 – Fishing-zone forecast (model demo)\n'
                'Test period: 2023–2024. Probabilities are model estimates, not a guarantee of catch.')
    with gr.Row():
        date_in = gr.Textbox(label='Forecast date (YYYY-MM-DD)', value='2023-06-15')
        lead_in = gr.Radio(['Day 1', 'Day 2', 'Day 3'], value='Day 1', label='How many days ahead')
        runs_in = gr.Slider(5, 50, value=30, step=5, label='MC dropout runs')
    btn = gr.Button('Predict', variant='primary')
    plot_out = gr.Plot(label='Maps')
    table_out = gr.Dataframe(label='How did the model do on this day?')
    btn.click(run_demo, [date_in, lead_in, runs_in], [plot_out, table_out])

demo.launch(share=True, show_error=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e0dd8c8bf49b0877d8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
